# 01 – Load, validate and analyse holdings

Loads the 55 holdings from the six pie sheets of the portfolio workbook into one clean table, validates it against the Portfolio sheet, analyses exposure, and reconciles the results with the workbook's exposure matrix.

## Setup

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
DATA_PATH = Path("../data/raw/portfolio_20261031.xlsx")
assert DATA_PATH.exists(), f"Workbook not found: {DATA_PATH.resolve()}"

## Load holdings

In [3]:
def read_table(path, sheet_name, header_label):
    """Read a table from a sheet whose header row starts with header_label.

    Finds the header row (first column equals header_label), re-reads the
    sheet from that row and cuts the table off at the first empty cell in
    the first column.
    """
    raw = pd.read_excel(path, sheet_name=sheet_name, header=None)
    header_row = raw[raw[0] == header_label].index[0]

    table = pd.read_excel(path, sheet_name=sheet_name, header=header_row)

    first_col = table.columns[0]
    first_empty = table.index[table[first_col].isna()][0]
    return table.iloc[:first_empty]

In [4]:
PIE_SHEETS = [
    "1) Kwaliteit",
    "2) Thematisch",
    "3) AI Boom",
    "4) Energie",
    "5) Mijnbouw",
    "6) Multipolaire Wereldorde",
]

CATEGORY_COLUMNS = {"Thema": "category", "Ketenlaag": "category", "Regio": "category"}

In [5]:
frames = []
for sheet_name in PIE_SHEETS:
    holdings = read_table(DATA_PATH, sheet_name, "Ticker")
    holdings = holdings.rename(columns=CATEGORY_COLUMNS)
    holdings["pie_sheet"] = sheet_name
    holdings["pie"] = sheet_name.split(") ", maxsplit=1)[1]
    frames.append(holdings)

len(frames)

6

In [6]:
all_holdings = pd.concat(frames, ignore_index=True)
all_holdings.shape

(55, 29)

In [7]:
COLUMN_NAMES = {
    "Ticker": "ticker",
    "Naam": "name",
    "ISIN": "isin",
    "Type": "asset_type",
    "Model": "model",
    "Weging pie": "weight_in_pie",
    "Weging portf.": "weight_in_portfolio",
    "Land": "country",
    "Beurs": "exchange",
    "Not. valuta": "listing_currency",
    "TER": "ter",
    "Replicatie": "replication",
    "Uitkering": "distribution",
    "USD": "econ_usd",
    "EUR": "econ_eur",
    "GBP": "econ_gbp",
    "CHF": "econ_chf",
    "CAD": "econ_cad",
    "Overig ontw.": "econ_other_dm",
    "Opkomend": "econ_em",
    "Controle valuta": "econ_check",
    "Hoofdvaluta": "main_currency",
    "Div. rendement": "dividend_yield",
    "Cijfer": "score",
    "Rating": "rating",
    "Rol: waarom deze positie": "role",
}

In [8]:
all_holdings = all_holdings.rename(columns=COLUMN_NAMES)

In [9]:
NUMERIC_COLUMNS = [
    "weight_in_pie",
    "weight_in_portfolio",
    "ter",
    "econ_usd",
    "econ_eur",
    "econ_gbp",
    "econ_chf",
    "econ_cad",
    "econ_other_dm",
    "econ_em",
    "dividend_yield",
    "score",
]

In [10]:
for col in NUMERIC_COLUMNS:
    all_holdings[col] = pd.to_numeric(all_holdings[col])

## Validation

In [11]:
TOLERANCE = 1e-9

In [12]:
pie_sums = all_holdings.groupby("pie")["weight_in_pie"].sum()
pie_check = (pie_sums - 1).abs() < TOLERANCE
pie_sums

pie
AI Boom                    1.0
Energie                    1.0
Kwaliteit                  1.0
Mijnbouw                   1.0
Multipolaire Wereldorde    1.0
Thematisch                 1.0
Name: weight_in_pie, dtype: float64

In [13]:
total_weight = all_holdings["weight_in_portfolio"].sum()
abs(total_weight - 1) < TOLERANCE

np.True_

In [14]:
targets = read_table(DATA_PATH, "Portfolio", "Pie")
targets = targets[targets["Pie"] != "Totaal"]
targets = targets.rename(columns={"Pie": "pie", "Doelgewicht": "target_weight", "Blok": "block", "Posities": "n_positions"})
targets["target_weight"] = pd.to_numeric(targets["target_weight"])
targets["n_positions"] = pd.to_numeric(targets["n_positions"])
targets[["pie", "target_weight", "block", "n_positions"]]

,pie,target_weight,block,n_positions
0,Kwaliteit,0.27,A,17.0
1,Thematisch,0.13,A,5.0
2,AI Boom,0.10,A,8.0
3,Energie,0.18,B,10.0
4,Mijnbouw,0.17,B,9.0
5,Multipolaire Wereldorde,0.15,B,6.0


In [15]:
actual = all_holdings.groupby("pie")["weight_in_portfolio"].sum()
target = targets.set_index("pie")["target_weight"]

target_check = (actual - target).abs() < TOLERANCE
target_check.all()

np.True_

In [16]:
ECON_COLUMNS = ["econ_usd", "econ_eur", "econ_gbp", "econ_chf", "econ_cad", "econ_other_dm", "econ_em"]
row_sums = all_holdings[ECON_COLUMNS].sum(axis=1)

actual_counts = all_holdings.groupby("pie")["ticker"].count()
target_counts = targets.set_index("pie")["n_positions"]

In [17]:
checks = {
    "total weight = 1": abs(total_weight - 1) < TOLERANCE,
    "pie weights sum to 1": pie_check.all(),
    "pie weights match targets": target_check.all(),
    "position counts match Portfolio": ((actual_counts - target_counts) == 0).all(),
    "no empty tickers": all_holdings["ticker"].notna().all(),
    "no empty ISINs": all_holdings["isin"].notna().all(),
    "unique tickers": not all_holdings["ticker"].duplicated().any(),
    "unique ISINs": not all_holdings["isin"].duplicated().any(),
    "currency split sums to 1 per row": ((row_sums - 1).abs() < TOLERANCE).all(),
}
validation = pd.Series(checks)
validation

total weight = 1                    True
pie weights sum to 1                True
pie weights match targets           True
position counts match Portfolio     True
no empty tickers                    True
no empty ISINs                      True
unique tickers                      True
unique ISINs                        True
currency split sums to 1 per row    True
dtype: bool

In [18]:
assert validation.all(), validation[~validation]

## Exposure

In [19]:
all_holdings = all_holdings.merge(
    targets[["pie", "block"]],
    on="pie",
    how="left",
    validate="many_to_one",
)
all_holdings.shape

(55, 30)

In [20]:
all_holdings.groupby("block")["weight_in_portfolio"].sum()

block
A    0.5
B    0.5
Name: weight_in_portfolio, dtype: float64

In [21]:
for col in ["pie", "asset_type", "country", "listing_currency"]:
    print(all_holdings.groupby(col)["weight_in_portfolio"].sum().sort_values(ascending=False).round(4))
    print()

pie
Kwaliteit                  0.27
Energie                    0.18
Mijnbouw                   0.17
Multipolaire Wereldorde    0.15
Thematisch                 0.13
AI Boom                    0.10
Name: weight_in_portfolio, dtype: float64

asset_type
Aandeel    0.5796
ETF        0.4204
Name: weight_in_portfolio, dtype: float64

country
Ierland        0.3871
VS             0.2945
Canada         0.0916
Zwitserland    0.0594
Luxemburg      0.0495
VK             0.0315
Frankrijk      0.0297
Nederland      0.0265
Zweden         0.0162
Taiwan         0.0140
Name: weight_in_portfolio, dtype: float64

listing_currency
USD    0.4658
EUR    0.3540
GBP    0.1089
CHF    0.0459
CAD    0.0254
Name: weight_in_portfolio, dtype: float64



In [22]:
weighted_fx = all_holdings[ECON_COLUMNS].mul(all_holdings["weight_in_portfolio"], axis=0)
weighted_fx["block"] = all_holdings["block"]

fx_by_block = weighted_fx.groupby("block").sum().T
fx_by_block

block,A,B
econ_usd,0.304691,0.184317
econ_eur,0.067891,0.014688
econ_gbp,0.008100,0.008478
econ_chf,0.045900,0.000000
econ_cad,0.013500,0.120190
econ_other_dm,0.045918,0.043702
econ_em,0.014000,0.128625


In [23]:
ECON_LABELS = {
    "econ_usd": "USD",
    "econ_eur": "EUR",
    "econ_gbp": "GBP",
    "econ_chf": "CHF",
    "econ_cad": "CAD",
    "econ_other_dm": "Other DM",
    "econ_em": "EM",
}

listing = all_holdings.groupby("listing_currency")["weight_in_portfolio"].sum()
economic = fx_by_block.sum(axis=1).rename(index=ECON_LABELS)

fx_compare = pd.DataFrame({"listing": listing, "economic": economic}).fillna(0)
fx_compare.round(4)

,listing,economic
CAD,0.0254,0.1337
CHF,0.0459,0.0459
EM,0.0000,0.1426
EUR,0.3540,0.0826
GBP,0.1089,0.0166
Other DM,0.0000,0.0896
USD,0.4658,0.4890


In [24]:
top10 = all_holdings.nlargest(10, "weight_in_portfolio")[["ticker", "pie", "weight_in_portfolio"]]
print(top10)
print(top10["weight_in_portfolio"].sum())

   ticker                      pie  weight_in_portfolio
40    GDX                 Mijnbouw               0.0544
30   WENS                  Energie               0.0486
49   EPIN  Multipolaire Wereldorde               0.0375
17   LOCK               Thematisch               0.0364
18   GRID               Thematisch               0.0351
19   WQTM               Thematisch               0.0286
50   AMEL  Multipolaire Wereldorde               0.0270
41   SILV                 Mijnbouw               0.0255
51    CHN  Multipolaire Wereldorde               0.0225
52   IUSS  Multipolaire Wereldorde               0.0225
0.33810000000000007


In [25]:
DRIVER_MAP = {
    "Elektrificatie & kernenergie": "Elektrificatie-gedreven",
    "Uranium": "Elektrificatie-gedreven",
    "Koper & basismetalen": "Elektrificatie-gedreven",
    "Strategische metalen": "Elektrificatie-gedreven",
    "Olie & gas": "Olie, gas en industriële gassen",
    "Midstream & LNG": "Olie, gas en industriële gassen",
    "Industriële gassen": "Olie, gas en industriële gassen",
    "Goud": "Monetair",
    "Zilver": "Monetair",
    "Royalty & streaming": "Monetair",
}

energy_mining = all_holdings[all_holdings["pie"].isin(["Energie", "Mijnbouw"])].copy()
energy_mining["driver"] = energy_mining["category"].map(DRIVER_MAP)
assert energy_mining["driver"].notna().all(), "Unmapped category in Energie/Mijnbouw"

drivers = energy_mining.groupby("driver")["weight_in_portfolio"].sum()
drivers

driver
Elektrificatie-gedreven            0.1240
Monetair                           0.1054
Olie, gas en industriële gassen    0.1206
Name: weight_in_portfolio, dtype: float64

## Reconciliation with exposure matrix

In [26]:
EXCEL_FX_NAMES = {
    "USD": "econ_usd",
    "EUR": "econ_eur",
    "GBP": "econ_gbp",
    "CHF": "econ_chf",
    "CAD": "econ_cad",
    "Overig ontwikkeld": "econ_other_dm",
    "Opkomend": "econ_em",
}

excel_fx = read_table(DATA_PATH, "Blootstellingsmatrix", "Valuta")
excel_fx = excel_fx[excel_fx["Valuta"].isin(EXCEL_FX_NAMES.keys())]
excel_fx = excel_fx.set_index("Valuta").rename(index=EXCEL_FX_NAMES)[["Blok A", "Blok B"]]
excel_fx.columns = ["A", "B"]
excel_fx = excel_fx.apply(pd.to_numeric)

fx_diff = (fx_by_block - excel_fx).abs()
(fx_diff < TOLERANCE).all().all()

np.True_

In [27]:
excel_drivers = read_table(DATA_PATH, "Blootstellingsmatrix", "Drijfveer")
excel_drivers = excel_drivers[excel_drivers["Drijfveer"] != "Totaal"]
excel_drivers = pd.to_numeric(excel_drivers.set_index("Drijfveer")["% portefeuille"])

((drivers - excel_drivers).abs() < TOLERANCE).all()

np.True_

In [28]:
all_holdings.nlargest(10, "weight_in_portfolio", keep="all")[["ticker", "weight_in_portfolio"]]

,ticker,weight_in_portfolio
40,GDX,0.0544
30,WENS,0.0486
49,EPIN,0.0375
17,LOCK,0.0364
18,GRID,0.0351
19,WQTM,0.0286
50,AMEL,0.0270
41,SILV,0.0255
51,CHN,0.0225
52,IUSS,0.0225


**Reconciliation result.** Sections 1 (economic currency by block) and 5 (Energy and Mining by driver) match the exposure matrix within 1e-9. Section 6 (top-10): the total (33.81%) matches, but ranks 9–10 differ because CHN, IUSS and LYXI are tied at 2.25%. pandas breaks ties by row order, Excel by its own formula logic. Not a data error; the top-10 is not unique with ties.

## Export

In [29]:
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

all_holdings.to_parquet(PROCESSED_DIR / "holdings.parquet", index=False)